Deploy the FinSight LLM Eval Pipeline to GitHub Actions

In [ ]:
import os
import json
import shutil
from pathlib import Path

# ── Secrets ──────────────────────────────────────────────────────
# === MANUAL SETUP (replace these) ===
GROQ_API_KEY = ''          # Optional for now
GITHUB_TOKEN = ''  # REQUIRED for GitHub
GITHUB_USERNAME = 'Syed ABjimiah'         # Your GitHub username

# Set environment variable
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# ── Create local repo directory ───────────────────────────────────
REPO_NAME = 'syed-finsight-llm-eval'
REPO_DIR = Path('C:/syed/uv_projects/day_13_github_task/Projects') / REPO_NAME

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

REPO_DIR.mkdir(parents=True)

# Sub-directories
(REPO_DIR / 'src').mkdir(exist_ok=True)
(REPO_DIR / 'tests').mkdir(exist_ok=True)
(REPO_DIR / '.github' / 'workflows').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'results').mkdir(exist_ok=True)

print(f'✅ Repo scaffold created at {REPO_DIR}')
print('\nDirectory structure:')
for p in sorted(REPO_DIR.rglob('*')):
    indent = ' ' * (len(p.relative_to(REPO_DIR).parts) - 1)
    print(f' {indent}{p.name}/' if p.is_dir() else f' {indent}{p.name}')

Step 1 — Write the Core Eval Harness Module (src/eval_harness.py)

In [ ]:
EVAL_HARNESS_PY = '''#!/usr/bin/env python3
"""
FinSight AI — LLM Evaluation Harness
Runs multi-model quality evaluation using Groq-hosted models.
"""

import os
import re
import json
import time
from groq import Groq
from collections import defaultdict
import statistics
from typing import List, Dict, Any

# ── Model registry ────────────────────────────────────────────────
GROQ_MODELS = {
    "llama-3.3-70b": "llama-3.3-70b-versatile",
    "llama-3.1-8b": "llama-3.1-8b-instant",
    "mixtral-8x7b": "mixtral-8x7b-32768",
    "gemma2-9b": "gemma2-9b-it",
}

# ── Pricing per 1M tokens (USD, approximate mid-2025) ─────────────
GROQ_PRICING = {
    "llama-3.3-70b-versatile": {"input": 0.59, "output": 0.79},
    "llama-3.1-8b-instant": {"input": 0.05, "output": 0.08},
    "mixtral-8x7b-32768": {"input": 0.24, "output": 0.24},
    "gemma2-9b-it": {"input": 0.20, "output": 0.20},
}

SYSTEM_PROMPT = """You are a credit analyst AI assistant at FinSight AI. Generate a concise credit risk memo (150-250 words) based on the provided borrower data. 
Structure: (1) Borrower Overview, (2) Key Financial Metrics, (3) Risk Assessment, (4) Recommendation. 
Use precise financial language. Do not fabricate or extrapolate data not provided."""

JUDGE_RUBRIC = """You are a senior credit risk officer evaluating AI-generated credit memos. Score on THREE dimensions (1-5 each).

FAITHFULNESS (1-5): Are all figures accurate and grounded in the source data?
COMPLETENESS (1-5): Does the memo cover borrower profile, metrics, risk, recommendation?
REGULATORY TONE (1-5): Is language precise, objective, and compliance-appropriate?

BORROWER DATA:
{borrower_data}

GENERATED MEMO:
{memo}

Respond ONLY with valid JSON (no markdown fences):
{{"faithfulness": , "completeness": , "regulatory_tone": , "reasoning": ""}}"""

CONSTRAINTS = {
    "hallucin_rate": 0.01,
    "latency_p95": 3.0,
    "avg_cost": 0.02,
}

def get_client() -> Groq:
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY environment variable not set")
    return Groq(api_key=api_key)

def compute_cost(model_id: str, input_tokens: int, output_tokens: int) -> float:
    p = GROQ_PRICING.get(model_id, {"input": 0.5, "output": 0.5})
    return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000

def call_groq(client: Groq, prompt: str, model_id: str, system: str = SYSTEM_PROMPT, max_tokens: int = 400) -> dict:
    start = time.time()
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=max_tokens,
        )
        latency = time.time() - start
        output = resp.choices[0].message.content
        in_tok = resp.usage.prompt_tokens
        out_tok = resp.usage.completion_tokens

        return {
            "output": output,
            "latency": round(latency, 3),
            "in_tokens": in_tok,
            "out_tokens": out_tok,
            "cost": round(compute_cost(model_id, in_tok, out_tok), 6),
            "error": None,
        }
    except Exception as e:
        return {
            "output": "", 
            "latency": round(time.time()-start, 3),
            "in_tokens": 0, 
            "out_tokens": 0, 
            "cost": 0,
            "error": str(e)
        }

def judge_memo(client: Groq, borrower_data: str, memo: str, judge_model: str = GROQ_MODELS["llama-3.3-70b"]) -> dict:
    prompt = JUDGE_RUBRIC.format(borrower_data=borrower_data, memo=memo)
    try:
        resp = client.chat.completions.create(
            model=judge_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300,
        )
        text = re.sub(r"```json|```|\n", "", resp.choices[0].message.content).strip()
        
        # More robust JSON parsing
        try:
            return json.loads(text)
        except:
            # Try to extract JSON if there's extra text
            match = re.search(r'\{.*\}', text, re.DOTALL)
            if match:
                return json.loads(match.group(0))
            raise
    except Exception as e:
        return {
            "faithfulness": None,
            "completeness": None,
            "regulatory_tone": None,
            "reasoning": f"ERROR: {e}"
        }

def check_hallucination(borrower_data: str, memo: str) -> dict:
    def _normalise(s: str):
        s = s.replace("$", "").replace(",", "").strip()
        if s.endswith("K"): return float(s[:-1]) * 1e3
        if s.endswith("M"): return float(s[:-1]) * 1e6
        if s.endswith("B"): return float(s[:-1]) * 1e9
        try:
            return float(s)
        except:
            return None

    src_vals = {_normalise(n) for n in re.findall(r'\$?[\d,]+\.?\d*[KMB]?', borrower_data) 
                if _normalise(n) is not None}
    memo_vals = {_normalise(n) for n in re.findall(r'\$?[\d,]+\.?\d*[KMB]?', memo) 
                 if _normalise(n) is not None and _normalise(n) > 0}

    hallucinated = [
        v for v in memo_vals 
        if v > 200 and not any(abs(v - s) / max(s, 0.01) < 0.05 for s in src_vals if s and s > 0)
    ]

    return {
        "hallucination_flag": len(hallucinated) > 0,
        "hallucination_count": len(hallucinated),
    }

def run_eval(test_cases: list, models: dict = None, judge: bool = True) -> list:
    if models is None:
        models = GROQ_MODELS

    client = get_client()
    results = []

    for tc in test_cases:
        for model_name, model_id in models.items():
            prompt = f"Generate a credit risk memo for the following borrower:\n\n{tc['data']}"
            result = call_groq(client, prompt, model_id)

            row = {
                "test_id": tc["id"],
                "difficulty": tc["difficulty"],
                "model": model_name,
                "model_id": model_id,
                "output": result["output"],
                "in_tokens": result["in_tokens"],
                "out_tokens": result["out_tokens"],
                "latency": result["latency"],
                "cost": result["cost"],
                "error": result["error"],
                "borrower_data": tc["data"],
            }

            if result["output"] and not result["error"]:
                h = check_hallucination(tc["data"], result["output"])
                row.update(h)

                if judge:
                    scores = judge_memo(client, tc["data"], result["output"])
                    row["score_faithfulness"] = scores.get("faithfulness")
                    row["score_completeness"] = scores.get("completeness")
                    row["score_regulatory_tone"] = scores.get("regulatory_tone")
                    row["judge_reasoning"] = scores.get("reasoning")

                    if all(row.get(k) is not None for k in ["score_faithfulness", "score_completeness", "score_regulatory_tone"]):
                        row["composite_score"] = round(
                            (row["score_faithfulness"] + row["score_completeness"] + row["score_regulatory_tone"]) / 3, 2
                        )

            time.sleep(0.2)  # Slightly increased for safety
            results.append(row)

    return results

def build_leaderboard(results: list) -> list:
    models = list({r["model"] for r in results})
    lb = []

    for model in models:
        rows = [r for r in results if r["model"] == model]
        latencies = sorted(r["latency"] for r in rows)
        p95_idx = max(0, int(len(latencies) * 0.95) - 1)

        composite_vals = [r["composite_score"] for r in rows if r.get("composite_score") is not None]
        hall_flags = [int(r.get("hallucination_flag", False)) for r in rows]

        entry = {
            "model": model,
            "composite": round(statistics.mean(composite_vals), 3) if composite_vals else None,
            "hallucin_rate": round(sum(hall_flags) / len(hall_flags), 3) if hall_flags else None,
            "latency_p95": round(latencies[p95_idx], 3),
            "avg_cost": round(statistics.mean(r["cost"] for r in rows), 6),
        }
        entry["meets_constraints"] = (
            entry["hallucin_rate"] is not None 
            and entry["hallucin_rate"] < CONSTRAINTS["hallucin_rate"]
            and entry["latency_p95"] < CONSTRAINTS["latency_p95"]
            and entry["avg_cost"] < CONSTRAINTS["avg_cost"]
        )
        lb.append(entry)

    return sorted(lb, key=lambda x: x.get("composite") or 0, reverse=True)

def check_quality_gate(leaderboard: list) -> dict:
    passing = [row for row in leaderboard if row.get("meets_constraints")]
    if passing:
        best = passing[0]["model"]
        return {
            "passed": True,
            "reason": f"Model '{best}' meets all FinSight production constraints.",
            "best_model": best
        }

    violations = []
    for row in leaderboard:
        if row.get("hallucin_rate") is not None and row["hallucin_rate"] >= CONSTRAINTS["hallucin_rate"]:
            violations.append(f"{row['model']}: hallucination {row['hallucin_rate']*100:.1f}%")

    return {
        "passed": False,
        "reason": "No model meets all constraints. Violations: " + "; ".join(violations[:2]),
        "best_model": None
    }
'''

with open(REPO_DIR / 'src' / 'eval_harness.py', 'w',encoding='utf-8') as f:
    f.write(EVAL_HARNESS_PY)

print('✅ src/eval_harness.py written')
print(f'   Lines: {len(EVAL_HARNESS_PY.splitlines())}')


In [ ]:
TEST_CASES_PY = '''"""
FinSight AI — Canonical 20 Test Cases
Used for LLM evaluation harness.
"""

TEST_CASES = [
    # EASY
    {"id": "TC01", "difficulty": "easy", "data": "Borrower: Apex Manufacturing Ltd. Revenue: $2.1M. Debt: $1.5M term loan (5 years)."},
    {"id": "TC02", "difficulty": "easy", "data": "Borrower: GreenLeaf Organics Inc. Revenue: $420K. Current ratio: 2.1x. No existing debt. Loan request: $800K for equipment financing."},
    {"id": "TC03", "difficulty": "easy", "data": "Borrower: Sunrise Hotels Group. Revenue: $3.9M. Existing debt: $2M for expansion."},
    {"id": "TC04", "difficulty": "easy", "data": "Borrower: TechBridge Solutions LLC. Annual Recurring Revenue (ARR): $500K. Seeking working capital loan."},
    {"id": "TC05", "difficulty": "easy", "data": "Borrower: Coastal Fisheries Co. Revenue: $1.4M. DSCR: 1.6x. Fleet value: $1.2M."},

    # MEDIUM
    {"id": "TC06", "difficulty": "medium", "data": "Borrower: RetailPro Chain. Revenue: $1.1M. Operating leases: $3M."},
    {"id": "TC07", "difficulty": "medium", "data": "Borrower: NovaBio Pharma. Pre-revenue startup. Raised $4M. Monthly burn: $150K. Requesting $1.5M bridge loan."},
    {"id": "TC08", "difficulty": "medium", "data": "Borrower: Atlas Construction. Revenue: $500K. Unresolved litigation. Loan request: $4M."},
    {"id": "TC09", "difficulty": "medium", "data": "Borrower: PrimeAgri Partners. Revenue: $900K. Farmland collateral valued at $1.8M."},
    {"id": "TC10", "difficulty": "medium", "data": "Borrower: Urban Mobility Inc. Monthly revenue: $350K. Negative EBITDA. Has $600K city contract."},

    # HARD
    {"id": "TC11", "difficulty": "hard", "data": "Borrower: GlobalTrade Ltd. Annual revenue: $6M. Complex international supply chain."},
    {"id": "TC12", "difficulty": "hard", "data": "Borrower: DataVault Systems. Revenue: $2.4M. Related-party transactions: $600K. Loan request: $2.5M."},
    {"id": "TC13", "difficulty": "hard", "data": "Borrower: Heritage Real Estate Fund. Net Asset Value (NAV): $8M."},
    {"id": "TC14", "difficulty": "hard", "data": "Borrower: CryptoAsset Ventures. Revenue: $4.2M from digital assets. Loan request: $1M."},
    {"id": "TC15", "difficulty": "hard", "data": "Borrower: MedDevice International. Revenue: $3.5M. High R&D spend."},

    # ADVERSARIAL (Designed to trigger hallucinations / poor reasoning)
    {"id": "TC16", "difficulty": "adversarial", "data": "Borrower: Pinnacle Energy. Revenue: $3.2M. Taxes paid: $800K (25% rate). EBITDA: $5.4M (no D&A). Loan request: $4M."},
    {"id": "TC17", "difficulty": "adversarial", "data": "Borrower: FreshFoods Co. Revenue: $7M. Q1 numbers submitted as representative of full year. Loan request: $2.5M."},
    {"id": "TC18", "difficulty": "adversarial", "data": "Borrower: TechStart Alpha. ARR: $300K. Financials show $3.6M ≠ $67K. Claims $1M ARR."},
    {"id": "TC19", "difficulty": "adversarial", "data": "Borrower: LuxProperty Group. Properties appraised at $18M. Stated LTV 65%, actual ~75%. Stressed ICR at +200bps: 0.85x. Loan: $3M."},
    {"id": "TC20", "difficulty": "adversarial", "data": "Borrower: NovaMed Clinic Group. Revenue: $1.8M (claimed 29% margin, industry typical 12-15%). No breakdown provided. No audited accounts. Loan request: $2M."},
]

# Smoke test — only 5 easy cases (used in CI for fast validation)
SMOKE_TEST_CASES = [t for t in TEST_CASES if t["difficulty"] == "easy"]

# For easy importing
__all__ = ["TEST_CASES", "SMOKE_TEST_CASES"]
'''

# Step 2 — Write the Test Data Module
with open(REPO_DIR / 'src' / 'test_cases.py', 'w', encoding='utf-8') as f:
    f.write(TEST_CASES_PY)

print('✅ src/test_cases.py written successfully')

In [ ]:
CI_RUNNER_PY = '''#!/usr/bin/env python3
"""
FinSight AI — CI Evaluation Runner
Invoked by GitHub Actions on every PR.
Exit code 0 = quality gate passed; Exit code 1 = gate failed.
"""

import sys
import json
import csv
import os
from pathlib import Path

# Add src/ to Python path
sys.path.insert(0, str(Path(__file__).parent))

from eval_harness import run_eval, build_leaderboard, check_quality_gate, GROQ_MODELS
from test_cases import SMOKE_TEST_CASES, TEST_CASES

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Config ────────────────────────────────────────────────────────
# Use smoke test (5 easy cases) in CI for speed; full suite on schedule
CI_MODE = os.environ.get("EVAL_MODE", "smoke")  # "smoke" | "full"

if CI_MODE == "smoke":
    test_cases = SMOKE_TEST_CASES
else:
    test_cases = TEST_CASES

# In CI, only run the two fastest/reliable models to save cost + time
CI_MODELS = {
    "llama-3.3-70b": GROQ_MODELS["llama-3.3-70b"],   # Strong judge-quality model
    "llama-3.1-8b": GROQ_MODELS["llama-3.1-8b"],     # Fast & cheap
}

print(f"[FinSight CI] Mode: {CI_MODE} | Cases: {len(test_cases)} | Models: {list(CI_MODELS.keys())}")


def save_results_csv(results: list, path: Path) -> None:
    if not results:
        return
    keys = list(results[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(results)
    print(f"✅ Saved: {path}")


def save_leaderboard_json(leaderboard: list, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(leaderboard, f, indent=2)
    print(f"✅ Saved: {path}")


def print_summary(leaderboard: list, gate: dict) -> None:
    print("\n" + "="*70)
    print("           FINSIGHT CI EVAL SUMMARY")
    print("="*70)
    
    header = f"{'Model':<20} {'Composite':>10} {'Hallucin%':>10} {'p95(s)':>8} {'Avg Cost':>10} {'Pass':>6}"
    print(header)
    print("-"*70)
    
    for row in leaderboard:
        composite = f"{row.get('composite', 'N/A'):.2f}" if row.get('composite') is not None else "N/A"
        hall = f"{row.get('hallucin_rate', 0)*100:.1f}%" if row.get('hallucin_rate') is not None else "N/A"
        p95 = f"{row.get('latency_p95', 0):.2f}" if row.get('latency_p95') is not None else "N/A"
        cost = f"${row.get('avg_cost', 0):.5f}" if row.get('avg_cost') is not None else "N/A"
        passed = "✅" if row.get("meets_constraints") else "❌"
        
        print(f"{row['model']:<20} {composite:>10} {hall:>10} {p95:>8} {cost:>10} {passed:>6}")
    
    print("="*70)
    status = "✅ PASSED" if gate["passed"] else "❌ FAILED"
    print(f"Quality Gate: {status}")
    print(f"Reason: {gate['reason']}")
    if gate.get("best_model"):
        print(f"Recommended model: {gate['best_model']}")
    print()


def main() -> int:
    print("[FinSight CI] Running evaluation harness...")
    
    results = run_eval(test_cases, models=CI_MODELS, judge=True)
    
    save_results_csv(results, RESULTS_DIR / "ci_eval_results.csv")
    leaderboard = build_leaderboard(results)
    save_leaderboard_json(leaderboard, RESULTS_DIR / "ci_leaderboard.json")
    
    gate = check_quality_gate(leaderboard)
    print_summary(leaderboard, gate)
    
    # Save gate outcome for GitHub Actions / downstream steps
    with open(RESULTS_DIR / "gate_outcome.json", "w", encoding="utf-8") as f:
        json.dump(gate, f, indent=2)
    
    # Append to GitHub Step Summary (if running in Actions)
    gh_summary = os.environ.get("GITHUB_STEP_SUMMARY")
    if gh_summary:
        try:
            with open(gh_summary, "a", encoding="utf-8") as f:
                f.write(f"## FinSight Eval — Quality Gate: {'✅ PASSED' if gate['passed'] else '❌ FAILED'}\n\n")
                f.write(f"**{gate['reason']}**\n\n")
                f.write("| Model | Composite | Hallucin% | p95 Latency | Avg Cost | Status |\n")
                f.write("|---|---|---|---|---|---|\n")
                for row in leaderboard:
                    composite = f"{row.get('composite', 'N/A'):.2f}" if row.get('composite') is not None else "N/A"
                    hall = f"{row.get('hallucin_rate', 0)*100:.1f}%" if row.get('hallucin_rate') is not None else "N/A"
                    p95 = f"{row.get('latency_p95', 0):.2f}s"
                    cost = f"${row.get('avg_cost', 0):.5f}"
                    icon = "✅" if row.get("meets_constraints") else "❌"
                    f.write(f"| {row['model']} | {composite} | {hall} | {p95} | {cost} | {icon} |\n")
        except Exception:
            pass  # Fail silently if summary writing fails
    
    return 0 if gate["passed"] else 1


if __name__ == "__main__":
    sys.exit(main())
'''

# Step 3 — Write the CI Runner Script
with open(REPO_DIR / 'src' / 'run_ci_eval.py', 'w', encoding='utf-8') as f:
    f.write(CI_RUNNER_PY)

print('✅ src/run_ci_eval.py written successfully')

In [ ]:
PYTEST_PY = '''"""
FinSight AI — pytest test suite for the eval pipeline.
Tests run in CI before the full eval to catch regressions quickly.
"""

import pytest
import os
import sys
from pathlib import Path

# Add src/ to Python path
sys.path.insert(0, str(Path(__file__).parent.parent / "src"))

from eval_harness import (
    check_hallucination,
    build_leaderboard,
    check_quality_gate,
    CONSTRAINTS,
    GROQ_MODELS,
)
from test_cases import TEST_CASES, SMOKE_TEST_CASES


# ── Unit tests: hallucination probe ──────────────────────────────
class TestHallucinationProbe:
    def test_no_hallucination_clean_data(self):
        """Model outputs only figures present in source data → no flag."""
        source = "Revenue: 2.1M. DSCR: 1.8x."
        memo = "The borrower reports revenue of 2.1M with DSCR of 1.8x."
        result = check_hallucination(source, memo)
        assert result["hallucination_flag"] is False, \
            f"Expected no hallucination, got count={result['hallucination_count']}"

    def test_hallucination_detected_fabricated_number(self):
        """Model invents a revenue figure not in source → flagged."""
        source = "Revenue: 5.2M and an EBITDA of 2.8M."
        memo = "Revenue is approximately $10.1M."   # Fabricated number
        result = check_hallucination(source, memo)
        assert result["hallucination_flag"] is True, "Expected hallucination flag"

    def test_small_numbers_ignored(self):
        """Ratios and small numbers (< 200) are not flagged as hallucinations."""
        source = "DSCR: 1.8x. Current ratio: 2.1"
        memo = "The DSCR is 1.8x indicating strong debt service capacity."
        result = check_hallucination(source, memo)
        assert result["hallucination_flag"] is False, \
            "Small ratio numbers should not be flagged"

    def test_tolerance_for_small_variation(self):
        """Small rounding differences should not trigger hallucination."""
        source = "Revenue: $2.1M"
        memo = "Revenue approximately 2,100,000"
        result = check_hallucination(source, memo)
        assert result["hallucination_flag"] is False


# ── Unit tests: leaderboard builder ──────────────────────────────
class TestLeaderboard:
    def _make_results(self, model_name, hall_flags, latencies, costs, composite_scores):
        return [
            {
                "model": model_name,
                "hallucination_flag": h,
                "latency": l,
                "cost": c,
                "composite_score": s,
            }
            for h, l, c, s in zip(hall_flags, latencies, costs, composite_scores)
        ]

    def test_leaderboard_sorted_by_composite(self):
        r1 = self._make_results("model-a", [False]*5, [0.5]*5, [0.001]*5, [4.5]*5)
        r2 = self._make_results("model-b", [False]*5, [0.4]*5, [0.001]*5, [3.5]*5)
        lb = build_leaderboard(r1 + r2)
        assert lb[0]["model"] == "model-a", "Higher composite should rank first"

    def test_meets_constraints_flag(self):
        r = self._make_results("good-model", [False]*5, [0.5]*5, [0.001]*5, [4.5]*5)
        lb = build_leaderboard(r)
        assert lb[0]["meets_constraints"] is True

    def test_fails_constraints_high_hallucination(self):
        r = self._make_results("bad-model", [True]*5, [0.5]*5, [0.001]*5, [4.0]*5)
        lb = build_leaderboard(r)
        assert lb[0]["meets_constraints"] is False, \
            "High hallucination rate should fail constraints"


# ── Unit tests: quality gate ──────────────────────────────────────
class TestQualityGate:
    def test_gate_passes_when_one_model_meets_all(self):
        leaderboard = [
            {
                "model": "good",
                "hallucin_rate": 0.005,
                "latency_p95": 1.2,
                "avg_cost": 0.001,
                "meets_constraints": True,
                "composite": 4.5
            }
        ]
        gate = check_quality_gate(leaderboard)
        assert gate["passed"] is True
        assert gate["best_model"] == "good"

    def test_gate_fails_when_no_model_meets_all(self):
        leaderboard = [
            {
                "model": "bad",
                "hallucin_rate": 0.05,
                "latency_p95": 4.0,
                "avg_cost": 0.03,
                "meets_constraints": False,
                "composite": 3.2
            }
        ]
        gate = check_quality_gate(leaderboard)
        assert gate["passed"] is False
        assert gate["best_model"] is None


# ── Sanity checks on test data ────────────────────────────────────
class TestTestData:
    def test_twenty_cases_exist(self):
        assert len(TEST_CASES) == 20

    def test_all_difficulties_represented(self):
        diffs = {t["difficulty"] for t in TEST_CASES}
        assert diffs == {"easy", "medium", "hard", "adversarial"}

    def test_smoke_set_is_easy_only(self):
        assert all(t["difficulty"] == "easy" for t in SMOKE_TEST_CASES)
        assert len(SMOKE_TEST_CASES) == 5

    def test_all_cases_have_required_keys(self):
        for tc in TEST_CASES:
            assert "id" in tc and "difficulty" in tc and "data" in tc


# ── Integration test (skipped by default) ─────────────────────────
@pytest.mark.skipif(
    not os.environ.get('RUN_INTEGRATION_TESTS'),
    reason="Integration tests skipped unless RUN_INTEGRATION_TESTS=1"
)
class TestIntegration:
    def test_eval_produces_passing_gate(self):
        """Full smoke eval — requires GROQ_API_KEY to be set."""
        from eval_harness import run_eval

        results = run_eval(
            SMOKE_TEST_CASES[:2],
            models={"llama-3.1-8b": GROQ_MODELS["llama-3.1-8b"]},
            judge=False
        )
        leaderboard = build_leaderboard(results)
        
        assert len(leaderboard) > 0
        assert leaderboard[0]["latency_p95"] < CONSTRAINTS["latency_p95"]
        assert leaderboard[0]["avg_cost"] < CONSTRAINTS["avg_cost"]
'''

# Step 4 — Write the pytest Test Suite
with open(REPO_DIR / 'tests' / 'test_eval_gates.py', 'w', encoding='utf-8') as f:
    f.write(PYTEST_PY)

print('✅ tests/test_eval_gates.py written successfully')

In [ ]:
WORKFLOW_YML = """# FinSight AI — LLM Evaluation Pipeline
# Triggers on PRs to main, pushes to main, and nightly schedule.

name: FinSight LLM Eval

on:
  pull_request:
    branches: [ main ]
    paths:
      - 'src/**'
      - 'tests/**'
      - '.github/workflows/llm-eval.yml'
  push:
    branches: [ main ]
  schedule:
    - cron: '0 2 * * *'   # Nightly at 02:00 UTC
  workflow_dispatch:
    inputs:
      eval_mode:
        description: 'Eval mode (smoke or full)'
        required: false
        default: 'smoke'
        type: choice
        options: [smoke, full]

jobs:
  # ─── Job 1: Fast Unit Tests ─────────────────────────────────────
  unit-tests:
    name: Unit Tests
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: pip

      - name: Install dependencies
        run: |
          pip install -q groq pytest

      - name: Run unit tests
        run: pytest tests/ -v --tb=short -m "not integration"

  # ─── Job 2: LLM Evaluation Quality Gate ────────────────────────
  llm-eval:
    name: LLM Eval Quality Gate
    runs-on: ubuntu-latest
    needs: unit-tests
    permissions:
      contents: read
      pull-requests: write
    env:
      GROQ_API_KEY: ${{ secrets.GROQ_API_KEY }}
      EVAL_MODE: ${{ github.event.inputs.eval_mode || (github.event_name == 'schedule' && 'full') || 'smoke' }}

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: pip

      - name: Install dependencies
        run: |
          pip install -q groq

      - name: Run FinSight Eval Harness
        id: run_eval
        run: python src/run_ci_eval.py
        continue-on-error: false

      - name: Upload evaluation artefacts
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: finsight-eval-results-${{ github.run_number }}
          path: results/
          retention-days: 30

      - name: Post results to PR comment
        if: github.event_name == 'pull_request'
        uses: actions/github-script@v7
        with:
          script: |
            const fs = require('fs');
            const gate = JSON.parse(fs.readFileSync('results/gate_outcome.json', 'utf8'));
            const icon = gate.passed ? '✅' : '❌';
            const status = gate.passed ? 'PASSED' : 'FAILED';
            
            const body = [
              '## ' + icon + ' FinSight LLM Eval — Quality Gate ' + status,
              '',
              '**' + gate.reason + '**',
              '',
              gate.best_model ? '🏆 Recommended model: `' + gate.best_model + '`' : '⚠️ No model met production constraints.',
              '',
              '> Full results available in [Actions Artifacts](' + 
              'https://github.com/' + context.repo.owner + '/' + context.repo.repo + 
              '/actions/runs/' + context.runId + ')'
            ].join('\\n');

            await github.rest.issues.createComment({
              owner: context.repo.owner,
              repo: context.repo.repo,
              issue_number: context.issue.number,
              body: body
            });

      - name: Fail job if quality gate failed
        if: steps.run_eval.outcome != 'success'
        run: |
          echo "❌ Quality gate failed!"
          cat results/gate_outcome.json
          exit 1
"""

# Step 5 — Write the GitHub Actions Workflow
with open(REPO_DIR / '.github' / 'workflows' / 'llm-eval.yml', 'w', encoding='utf-8') as f:
    f.write(WORKFLOW_YML)

print('✅ .github/workflows/llm-eval.yml written successfully')